# Memory

- LLM과 주고 받은 대화를 저장하는 기능을 말한다. 
  - LLM 모델은 대화의 상태를 저장하지 않는다. 그래서 질문을 하면 그것에 대한 답변을 하고 끝이다. 대화 내용에 따라 이전 대화 내용을 바탕으로 연결되는 질문을 하고 그것에 대한 답변을 받아야 할 때가 있다. 이런 경우 지금 까지의 대화 내용을 저장하는 것을 메모리(memory)라고 한다.
- **방식**
  - 대화 내용을 저장한 뒤 다음 질문을 할 때 저장된 이전 질문들을 합쳐서 전송한다.
  - 이전 대화내용을 어떻게 저장하는지에 따라 다양한 방식의 memory 기능이 있다.
    - LLM은 [입력 토큰수의 제한이](https://platform.openai.com/docs/models) 있기 때문에 대화를 무한정 저장할 수없다.
    - Langchain은 이전 대화 내용들을 요약하거나 최신 몇개만 저장하는 방식의 다양한 memory 방식을 제공한다.

![memory.png](figures/memory.png)

## 주요 Memory Class
### ConversationBufferMemory
- https://python.langchain.com/v0.1/docs/modules/memory/types/buffer/
- **대화를 모두 그대로 저장**한다.
- 대화가 길어질 경우 저장 양이 많아지는 문제가 있다.

### ConversationBufferWindowMemory
- **최신 대화 K개만 저장** 하고 그 이전 대화는 삭제한다. (K를 window size라고 하고 객체 생성시 설정한다.)
  - 한개의 대화는 input(질문)-output(답변) 한 쌍을 말한다.
    
### ConversationTokenBufferMemory
- https://python.langchain.com/v0.1/docs/modules/memory/types/token_buffer/
- 지정한 **token 수를 넘지 않는 범위**에서 최신 대화들을 저장한다.
  

### ConversationSummaryMemory
- https://python.langchain.com/v0.1/docs/modules/memory/types/summary/
- **기존 대화들을 요약**해서 저장한다. 다만 요약은 llm이 하기 때문에 객체 생성시 llm 모델을 지정해야 한다.
- 대화 내용이 요약되어 전송되므로 토큰 수를 줄여 요금을 절약할 수있다.
  
### ConversationSummaryBufferMemory
- https://python.langchain.com/v0.1/docs/modules/memory/types/summary_buffer/
- 대화들 자체를 메모리에 저장하다가 지정한 token을 넘어가면 **오래된 대화들 순서대로 요약**한다. 
  - 최신대화는 그대로 저장하고 오래된 대화는 요약해서 저장하는 방식.

> #### ConversationSummaryMemory 토큰 비용
> - 초기 비용은 ConversationSummaryMemory가 기존 대화를 요약하기 위해 LLM을 사용하므로 ConversationBufferMemory보다 더 많은 토큰을 사용한다. 그래서 비용이 더 많이 든다.
> - 대화가 길어질수록 ConversationBufferMemory모든 대화 내용을 그대로 저장하므로 토큰 수가 선형적으로 증가한다. 
> - 반면 ConversationSummaryMemory는 대화가 길어질수록 요약된 형태로 기존 대화들이 저장되어 토큰 증가율이 더 낮아지게 되서 비용을 절감할 수있다.
> - 비용과 관련해 **짧은 대화**일 경우 ConversationBufferMemory 가 효율적이고 **긴 대화일 경우** ConversationSummaryMemory가 효율적이다.
> - **요약 비용을 절감** 하기 위해 **요약에는 저렴한 모델**을 사용할 수 있다.

> #### Deprecated
> - 위 메모리 저장 방식은 0.3.1 부터 deprecated 되었다. (1.0 버전에서 제거될 예정) 
> - 대신 RunnableWithMessageHistory 사용이 권장 된다.

### 공통 메소드
- **initializer**
  - memory_key: str = "history"
    - 대화내역을 dict로 저장하는데 그때 사용되는 key.
    - default는 **"history"**
    - Prompt template에서 memory의 대화내용을 저장하는 placeholder(template variable)의 이름을 이 memory_key 로 지정한다.
  - return_message: bool
    - True: 각 대화내용을 Message(HumanMessage, AIMessage) 객체에 저장하고 그것을 List로 묶어서 반환
    - False: 대화내용을 문자열(str)로 반환한다.
  - chat_memory: BaseChatMessageHistory
    - 대화 history를 어디에 저장할 지 설정. (메모리, sql등)
- **save_context(inputs: dict, outputs: dict)**
  - Memory에 대화내용(Context)을 저장.
  - 파라미터
    - inputs: Human message
    - outputs: AI message
- **load_memory_variables(dict)**
  - 저장된 대화내용(context)를 반환.
  - argument로 빈 dict를 넣어준다.
- **clear()**
  - 저장된 모든 대화 비우기


## 대화 저장

In [2]:
from dotenv import load_dotenv
load_dotenv()

from langchain.memory import ConversationBufferMemory
from pprint import pprint

# 메모리 객체 생성
memory = ConversationBufferMemory(
    memory_key = "chat_history",
    return_messages = True, # 대화 내용을 message 객체로 반환(False: 문자열)
)

# 메모리에 대화 저장
memory.save_context(
    inputs = {"human":'안녕하세요. 은행 계좌를 개설하고싶습니다.'},
    outputs = {'ai':'개좌 개설을 원하시는 군요. 신분증을 준비해주세요'}
)

#저장된 대화들을 조회
save_conv = memory.load_memory_variables({})
pprint(save_conv)

{'chat_history': [HumanMessage(content='안녕하세요. 은행 계좌를 개설하고싶습니다.', additional_kwargs={}, response_metadata={}),
                  AIMessage(content='개좌 개설을 원하시는 군요. 신분증을 준비해주세요', additional_kwargs={}, response_metadata={})]}


In [3]:
save_conv['chat_history']

[HumanMessage(content='안녕하세요. 은행 계좌를 개설하고싶습니다.', additional_kwargs={}, response_metadata={}),
 AIMessage(content='개좌 개설을 원하시는 군요. 신분증을 준비해주세요', additional_kwargs={}, response_metadata={})]

In [4]:
memory.save_context(
    inputs = {'human':'신분증을 준비했습니다. 다음에는 무엇을 하면 될까요?'},
    outputs = {'ai':'신분증 앞면을 촬영해서 업로드 해주세요'}
)

pprint(memory.load_memory_variables({}))

{'chat_history': [HumanMessage(content='안녕하세요. 은행 계좌를 개설하고싶습니다.', additional_kwargs={}, response_metadata={}),
                  AIMessage(content='개좌 개설을 원하시는 군요. 신분증을 준비해주세요', additional_kwargs={}, response_metadata={}),
                  HumanMessage(content='신분증을 준비했습니다. 다음에는 무엇을 하면 될까요?', additional_kwargs={}, response_metadata={}),
                  AIMessage(content='신분증 앞면을 촬영해서 업로드 해주세요', additional_kwargs={}, response_metadata={})]}


In [5]:
memory.clear() # 저장 내용 모두 삭제
pprint(memory.load_memory_variables({}))

{'chat_history': []}


# 주요 메모리 클래스 예제

## ConversationBufferMemory
메모리에 대화를 모두 저장한다.

In [6]:
from langchain.memory import ConversationBufferMemory
from pprint import pprint

memory = ConversationBufferMemory(
    return_messages=True
)

memory.save_context(
    inputs={
        "human":"안녕하세요"
    },
    outputs={
        "ai": "반갑습니다."
    }
)

memory.save_context(
    inputs={
        "human":"한국 여행에 대해 물어보려고 합니다."
    },
    outputs={
        "ai":"무엇이든 물어보세요."
    }
)

memory.save_context(
    inputs={
        "human":"여행 할 때 꼭 먹어봐야할 음식 3개만 추천해줘."
    },
    outputs={
        "ai":"불고기, 비빔밥, 삼겹살을 추천합니다."
    }
)

memory.save_context(
    inputs={
        "human":"여행지 두 곳을 추천해줘."
    },
    outputs={
        "ai":"경복궁과 국립중앙박물관입니다."
    }
)
memory.save_context(
    inputs={
        "human":"여행지 한 곳 더 추천해줘."
    },
    outputs={
        "ai":"민속촌을 추천합니다."
    }
)

memory.save_context(
    inputs={
        "human":"한국의 자연을 느낄 수있는 여행지를 알려줘."
    },
    outputs={
        "ai":"북한산을 추천합니다. 서울 도심에서 자연을 느낄 수있습니다."
    }
)

pprint(memory.load_memory_variables({}))

{'history': [HumanMessage(content='안녕하세요', additional_kwargs={}, response_metadata={}),
             AIMessage(content='반갑습니다.', additional_kwargs={}, response_metadata={}),
             HumanMessage(content='한국 여행에 대해 물어보려고 합니다.', additional_kwargs={}, response_metadata={}),
             AIMessage(content='무엇이든 물어보세요.', additional_kwargs={}, response_metadata={}),
             HumanMessage(content='여행 할 때 꼭 먹어봐야할 음식 3개만 추천해줘.', additional_kwargs={}, response_metadata={}),
             AIMessage(content='불고기, 비빔밥, 삼겹살을 추천합니다.', additional_kwargs={}, response_metadata={}),
             HumanMessage(content='여행지 두 곳을 추천해줘.', additional_kwargs={}, response_metadata={}),
             AIMessage(content='경복궁과 국립중앙박물관입니다.', additional_kwargs={}, response_metadata={}),
             HumanMessage(content='여행지 한 곳 더 추천해줘.', additional_kwargs={}, response_metadata={}),
             AIMessage(content='민속촌을 추천합니다.', additional_kwargs={}, response_metadata={}),
             HumanMessage(content='한국의 

In [7]:
# llm 모델에 요청
from langchain.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser

model = ChatOpenAI(model='gpt-4o-mini')
prompt_template = ChatPromptTemplate(
    [
        ('system', '당신은 한국 여행 전문가입니다. 한국 여행과 관련된 다양한 정보를 알려주세요'),
        MessagesPlaceholder('history', optional=True), # 저장 내용 기억
        ('human', '{query}')
    ]
)
chain = prompt_template | model | StrOutputParser()


In [8]:
query = '마지막 소개한 여행지로 가는 방법을 알려주세요'
res = chain.invoke({'query':query, 'history':memory.load_memory_variables({})['history']})

print(res)

북한산으로 가는 방법은 다음과 같습니다:

1. **대중교통 이용하기**:
   - **지하철**: 
     - 3호선 **경복궁역**에서 하차 후, 3번 출구로 나와 버스를 이용하거나, 북한산국립공원 입구로 도보 이동할 수 있습니다.
     - 6호선 **화정역**에서 하차 후, 1번 출구로 나와서 버스를 이용하여 북한산 입구로 이동할 수 있습니다.
   - **버스**: 
     - **7016, 7211, 1020** 등의 버스를 이용하여 북한산국립공원 입구로 갈 수 있습니다.

2. **자동차 이용하기**:
   - 네비게이션에 '북한산국립공원' 또는 '북한산 입구'를 입력 후, 주차공간에 주차 후 하이킹을 시작할 수 있습니다.

북한산에서는 다양한 등산로가 있으니, 자신의 체력에 맞는 코스를 선택하여 즐기시면 좋습니다!


In [9]:
# 최신 대화를 memory에 저장
memory.save_context(
    inputs={'human':query},
    outputs={'ai':res}
)

pprint(memory.load_memory_variables({})['history'])

[HumanMessage(content='안녕하세요', additional_kwargs={}, response_metadata={}),
 AIMessage(content='반갑습니다.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='한국 여행에 대해 물어보려고 합니다.', additional_kwargs={}, response_metadata={}),
 AIMessage(content='무엇이든 물어보세요.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='여행 할 때 꼭 먹어봐야할 음식 3개만 추천해줘.', additional_kwargs={}, response_metadata={}),
 AIMessage(content='불고기, 비빔밥, 삼겹살을 추천합니다.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='여행지 두 곳을 추천해줘.', additional_kwargs={}, response_metadata={}),
 AIMessage(content='경복궁과 국립중앙박물관입니다.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='여행지 한 곳 더 추천해줘.', additional_kwargs={}, response_metadata={}),
 AIMessage(content='민속촌을 추천합니다.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='한국의 자연을 느낄 수있는 여행지를 알려줘.', additional_kwargs={}, response_metadata={}),
 AIMessage(content='북한산을 추천합니다. 서울 도심에서 자연을 느낄 수있습니다.', addition

## ConversationBufferWindowMemory
최신 대화 K개만 저장 한다.

In [10]:
def save(memory) :
    memory.save_context(
        inputs={
            "human":"안녕하세요"
        },
        outputs={
            "ai": "반갑습니다."
        }
    )

    memory.save_context(
        inputs={
            "human":"한국 여행에 대해 물어보려고 합니다."
        },
        outputs={
            "ai":"무엇이든 물어보세요."
        }
    )

    memory.save_context(
        inputs={
            "human":"여행 할 때 꼭 먹어봐야할 음식 3개만 추천해줘."
        },
        outputs={
            "ai":"불고기, 비빔밥, 삼겹살을 추천합니다."
        }
    )

    memory.save_context(
        inputs={
            "human":"여행지 두 곳을 추천해줘."
        },
        outputs={
            "ai":"경복궁과 국립중앙박물관입니다."
        }
    )
    memory.save_context(
        inputs={
            "human":"여행지 한 곳 더 추천해줘."
        },
        outputs={
            "ai":"민속촌을 추천합니다."
        }
    )

    memory.save_context(
        inputs={
            "human":"한국의 자연을 느낄 수있는 여행지를 알려줘."
        },
        outputs={
            "ai":"북한산을 추천합니다. 서울 도심에서 자연을 느낄 수있습니다."
        }
    )

In [11]:
from langchain.memory import ConversationBufferWindowMemory
from warnings import filterwarnings
filterwarnings('ignore')

memory = ConversationBufferWindowMemory(
    k=5, # 최근 대화 (input+output : 1개)
    return_messages=True
)

save(memory)

In [12]:
from pprint import pprint
pprint(memory.load_memory_variables({})['history'])

[HumanMessage(content='한국 여행에 대해 물어보려고 합니다.', additional_kwargs={}, response_metadata={}),
 AIMessage(content='무엇이든 물어보세요.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='여행 할 때 꼭 먹어봐야할 음식 3개만 추천해줘.', additional_kwargs={}, response_metadata={}),
 AIMessage(content='불고기, 비빔밥, 삼겹살을 추천합니다.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='여행지 두 곳을 추천해줘.', additional_kwargs={}, response_metadata={}),
 AIMessage(content='경복궁과 국립중앙박물관입니다.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='여행지 한 곳 더 추천해줘.', additional_kwargs={}, response_metadata={}),
 AIMessage(content='민속촌을 추천합니다.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='한국의 자연을 느낄 수있는 여행지를 알려줘.', additional_kwargs={}, response_metadata={}),
 AIMessage(content='북한산을 추천합니다. 서울 도심에서 자연을 느낄 수있습니다.', additional_kwargs={}, response_metadata={})]


## ConversationTokenBufferMemory
대화의 토큰 길이를 기준으로 최신 대화만 저장한다.

In [13]:
from langchain.memory import ConversationTokenBufferMemory
from langchain_openai import ChatOpenAI

model = ChatOpenAI(model='gpt-4o-mini')
memory = ConversationTokenBufferMemory(
    llm=model, # llm 모델 필요 (토큰 체크용 - 모델마다 토큰 인식 다르게함)
    max_token_limit=100,
    return_messages=True
)

save(memory)

pprint(memory.load_memory_variables({}))

{'history': [AIMessage(content='경복궁과 국립중앙박물관입니다.', additional_kwargs={}, response_metadata={}),
             HumanMessage(content='여행지 한 곳 더 추천해줘.', additional_kwargs={}, response_metadata={}),
             AIMessage(content='민속촌을 추천합니다.', additional_kwargs={}, response_metadata={}),
             HumanMessage(content='한국의 자연을 느낄 수있는 여행지를 알려줘.', additional_kwargs={}, response_metadata={}),
             AIMessage(content='북한산을 추천합니다. 서울 도심에서 자연을 느낄 수있습니다.', additional_kwargs={}, response_metadata={})]}


## ConversationSummaryMemory
대화를 요약해서 저장한다.

In [14]:
from langchain.memory import ConversationSummaryMemory

memory = ConversationSummaryMemory(
    llm=model, # 대화 내용 요약할 모델 설정
)

save(memory)

pprint(memory.load_memory_variables({})['history'])

print(memory.prompt) # llm에게 대화를 요청하기 위한 prompt template 조회

('The human greets the AI with "안녕하세요," and the AI responds with "반갑습니다." The '
 'human then mentions they want to ask about traveling to Korea, and the AI '
 'invites them to ask anything. The human requests recommendations for three '
 'must-try foods when traveling, and the AI suggests bulgogi, bibimbap, and '
 'samgyeopsal. The human then asks for recommendations for two travel '
 'destinations, and the AI suggests Gyeongbokgung Palace and the National '
 'Museum of Korea. Finally, the human asks for one more travel destination, '
 'and the AI recommends the Korean Folk Village. The human then asks for a '
 "travel destination where they can experience Korea's natural beauty, and the "
 'AI recommends Bukhansan, which provides a natural escape near downtown '
 'Seoul.')
input_variables=['new_lines', 'summary'] input_types={} partial_variables={} template='Progressively summarize the lines of conversation provided, adding onto the previous summary returning a new summary.\n\nEXAMPLE

## ConversationSummaryBufferMemory
오래된 대화는 요약하고 최신대화는 그대로 저장한다.

In [15]:
from langchain.memory import ConversationSummaryBufferMemory

memory = ConversationSummaryBufferMemory(
    llm=model, # 요약 모델
    max_token_limit=100,
    return_messages=True
)

save(memory)

pprint(memory.load_memory_variables({}))

{'history': [SystemMessage(content='The human greets the AI in Korean, and the AI responds warmly, encouraging the human to ask anything. The human expresses a desire to ask about traveling in Korea and requests recommendations for three must-try foods while traveling. The AI recommends bulgogi, bibimbap, and samgyeopsal. The human then asks the AI to recommend two travel destinations.', additional_kwargs={}, response_metadata={}),
             AIMessage(content='경복궁과 국립중앙박물관입니다.', additional_kwargs={}, response_metadata={}),
             HumanMessage(content='여행지 한 곳 더 추천해줘.', additional_kwargs={}, response_metadata={}),
             AIMessage(content='민속촌을 추천합니다.', additional_kwargs={}, response_metadata={}),
             HumanMessage(content='한국의 자연을 느낄 수있는 여행지를 알려줘.', additional_kwargs={}, response_metadata={}),
             AIMessage(content='북한산을 추천합니다. 서울 도심에서 자연을 느낄 수있습니다.', additional_kwargs={}, response_metadata={})]}


# Chain
## Off-the-shelf chains

In [18]:
from langchain.chains import LLMChain, ConversationChain  # 미리 정의되어있는 Chain class
from langchain.memory import ConversationSummaryBufferMemory
from langchain.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from pprint import pprint
from warnings import filterwarnings
from dotenv import load_dotenv

filterwarnings('ignore')
load_dotenv()

prompt_template = ChatPromptTemplate(
    [
        ('system', '답변은 3문장 이내로 간결하게 표현해줘'),
        MessagesPlaceholder('history'),  # variable_name -> memory key와 동일한 값으로 설정
        ('human', '{query}')
    ]
)

# 대화 내용을 생성하는 모델과 요약하는 모델 따로 지정 가능
model = ChatOpenAI(
    model='gpt-4o-mini'
)

memory = ConversationSummaryBufferMemory(
    llm=model,
    max_token_limit=200,
    memory_key='history',  # messageplaceholder 이름과 같아야함
    return_messages=True
)
output_parser = StrOutputParser()

chain = LLMChain(
    llm=model,
    prompt=prompt_template,
    output_parser=output_parser,
    memory=memory # history 추가
)

result = chain.invoke({'query':'안녕하세요'}) # template 변수에 넣은 값들 key=value, history에 저장된 값들, 응답 값을 dictionary에 묶어서 반환
print(result)
print(result['text'])


{'query': '안녕하세요', 'history': [HumanMessage(content='안녕하세요', additional_kwargs={}, response_metadata={}), AIMessage(content='안녕하세요! 무엇을 도와드릴까요?', additional_kwargs={}, response_metadata={})], 'text': '안녕하세요! 무엇을 도와드릴까요?'}
안녕하세요! 무엇을 도와드릴까요?


In [19]:
memory.load_memory_variables({})['history']

[HumanMessage(content='안녕하세요', additional_kwargs={}, response_metadata={}),
 AIMessage(content='안녕하세요! 무엇을 도와드릴까요?', additional_kwargs={}, response_metadata={})]

In [60]:
query = input('질문 : ')
while True :
    if query == 'q!' :
        break;
    result = chain.invoke({'query':query})
    print('Human', query)
    print('AI', result['text'])
    print()
    query = input('질문 : ')

Human 미국 여행지에 대해서 설명해줘
AI 미국은 다양한 여행지가 있습니다. 뉴욕의 타임스 스퀘어, 로스앤젤레스의 할리우드, 그리고 그랜드 캐년의 자연 경관은 특히 유명합니다. 각 지역은 독특한 문화와 경험을 제공하므로 방문할 가치가 충분합니다.

Human 마지막 여행지에 대해서 설명해줘
AI 마지막 여행지는 일본의 교토입니다. 이곳은 전통적인 사원과 정원이 아름답게 보존되어 있어 역사적인 매력이 가득합니다. 또한, 맛있는 음식과 독특한 문화 체험이 가능해 많은 관광객들이 찾는 곳입니다.

Human 마지막 여행지 그랜드 캐년이였는데?
AI 그랜드 캐년은 웅장한 자연 경관과 다양한 하이킹 코스로 유명합니다. 특히 일출과 일몰 때의 경치는 압도적입니다. 방문할 때는 충분한 준비와 안전을 고려하는 것이 중요합니다.

Human 내가 아까 뭐라고 물어봤지?
AI 죄송하지만, 이전 대화 내용을 기억할 수 없습니다. 질문을 다시 해주시면 도움을 드리겠습니다.



In [20]:
pprint(memory.load_memory_variables({})['history'])

[HumanMessage(content='안녕하세요', additional_kwargs={}, response_metadata={}),
 AIMessage(content='안녕하세요! 무엇을 도와드릴까요?', additional_kwargs={}, response_metadata={})]


## LCEL

In [21]:
from langchain_core.runnables import RunnablePassthrough, chain

prompt_template = ChatPromptTemplate(
    [
        ('system', '답변은 세 문장 이하로 간결하게 대답해줘'),
        MessagesPlaceholder('history'),
        ('human', '{query}')
    ]
)

model = ChatOpenAI(model='gpt-4o-mini')
output_parser = StrOutputParser()
memory = ConversationBufferMemory(
    llm=model,
    max_token_limit=200,
    return_messages=True,
    memory_key='history'
)

In [22]:
def load_memory(input) :
    # RunnablePassThourgh.assign(key=함수)
    # input : RannablePassThrough.invoke(입력값)의 입력값을 받는 파라미터
    ###### memory에서 저장된 대화 내역을 반환하는 함수
    return memory.load_memory_variables({})['history']


@chain
def question_chain(query) :
    # chain.invoke({'query':query})
    # 입력 : {'query':query} -> {'query':query, 'history':load_memory()}
    ### 메모리의 history를 추가해서 프롬프트에 전달 (RunnablePassThrough.assign())
    chain = (RunnablePassthrough.assign(history=load_memory) | prompt_template | model | output_parser)
    #chain을 이용해 질문 요청 -> 응답
    result = chain.invoke({'query':query})
    # 질문과 응답을 메모리에 저장 - 문자열로 저장
    memory.save_context(inputs={'human':query}, outputs={'ai':result})
    return result

result = question_chain.invoke('미국의 유명한 위인 세 명을 알려주세요')
print(result)

조지 워싱턴, 에이브러햄 링컨, 마틴 루터 킹 주니어가 미국의 유명한 위인들입니다.


In [24]:
result = question_chain.invoke('두번째 사람은 언제 태어났니')
result

'에이브러햄 링컨은 1809년 2월 12일에 태어났습니다.'